* Script para sacar los datos del al dataset

In [2]:
import requests
import pandas as pd
import time
import os

# Carpeta donde guardaremos el CSV
os.makedirs("../data", exist_ok=True)

# Resource oficial 2025
RESOURCE_ID = "510d138a-6c6d-4dce-8d30-8c77de58d787"
CKAN_URL = "https://catalogodatos.cnmc.es"
HEADERS = {"User-Agent": "IA-Proyecto-OPEC"}

def descargar_recurso_completo(resource_id, limit=32000):
    registros = []
    offset = 0

    while True:
        url = f"{CKAN_URL}/api/3/action/datastore_search"
        params = {
            "resource_id": resource_id,
            "limit": limit,
            "offset": offset
        }
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        data = r.json()["result"]["records"]
        if not data:
            break
        registros.extend(data)
        offset += limit
        print(f"⬇ Descargados {len(registros)} registros hasta ahora...")
        time.sleep(0.2)

    return registros

# Descargar todos los datos del recurso
datos = descargar_recurso_completo(RESOURCE_ID)

# Convertir a DataFrame
df = pd.DataFrame(datos)

# Limpiar espacios de los nombres de columnas
df.columns = df.columns.str.strip()

# Revisar nombres de columnas
print("Columnas del DataFrame:", df.columns.tolist())

# Renombrar columnas correctamente
df.rename(columns={
    "fecha_precio": "fecha",
    "provincia": "provincia",
    "producto": "producto",
    "promedio_de_pai_diario_cubo": "pai",
    "promedio_de_pvp_diario_cubo": "pvp"
}, inplace=True)

# Convertir tipos correctamente
df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d")
df["pai"] = pd.to_numeric(df["pai"], errors="coerce")
df["pvp"] = pd.to_numeric(df["pvp"], errors="coerce")

# Eliminar filas con nulos
df.dropna(inplace=True)

# Guardar CSV final limpio
df.to_csv("../data/precios_petroleo_2025_limpio.csv", index=False)
print(f"✅ Datos guardados en '../data/precios_petroleo_2025_limpio.csv' ({len(df)} registros)")


⬇ Descargados 32000 registros hasta ahora...
⬇ Descargados 64000 registros hasta ahora...
⬇ Descargados 69107 registros hasta ahora...
Columnas del DataFrame: ['_id', 'fecha_precio', 'provincia', 'producto', 'promedio_de_pai_diario_cubo', 'promedio_de_pvp_diario_cubo']
✅ Datos guardados en '../data/precios_petroleo_2025_limpio.csv' (69107 registros)
